# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 13 · Fixed-estimator temporal ablation

Compare the **72-column control** with **control + 62 direct-state fields** on the two existing later folds. Candidate definitions, estimator settings, seed, row ordering and target remain fixed. Columns are screened on each fold’s training rows only. At most eight new coordinate models.

A negative feature result is useful evidence. The inspected discovery fold cannot rescue a failed later-only result.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess, signal
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round5')
OUT = Path('/home/sagemaker-user/nfl-feature-round5-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Use the existing NFL space and upload/extract the Round 5 package first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage, fold=None):
    command = [str(PY), str(KIT/'run_round.py'), stage]
    if fold is not None: command += ['--fold', str(fold)]
    env = os.environ.copy()
    env.update({'OMP_NUM_THREADS':'2','OPENBLAS_NUM_THREADS':'2','PYTHONDONTWRITEBYTECODE':'1'})
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=env)
    try:
        for line in process.stdout: print(line, end='')
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try: process.wait(timeout=12)
        except subprocess.TimeoutExpired: process.terminate()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped (exit {code}). Preserve checkpoints and export the report. Do not alter settings.')
def show(fig, name):
    visuals.save(fig, OUT, name).show()
def receipt(name):
    return visuals.load(OUT/name)

assert receipt('preparation.json')['status'] == 'temporal_state_ready'

## 1. Fold 2 — four new coordinate models at most

Save each 30-iteration checkpoint. The fixed final exposure is 120 iterations. A 5% or larger deterioration triggers an early cost stop before fold 3. This is a futility rule, not a new acceptance threshold.

In [ ]:
run('fit', fold=2)
f2 = receipt('fold_2/summary.json')
assert f2['status'] == 'temporal_fold_complete'
print(json.dumps({'metrics':f2['metrics'],'comparison':f2['comparison'],'new_models':f2['new_coordinate_models']}, indent=2))

## 2. Fold 3 — conditional on the declared cost guard

A modest negative fold is retained and the second later fold still runs; do not hide variability. A severe negative fold stops new fitting. Do not tune or change the cutoff after seeing fold 2.

In [ ]:
if f2['comparison']['relative_gain'] <= -0.05:
    print('FUTILITY STOP: no fold 3 fit. Continue only to replay/report cells below.')
    run('fit', fold=3)  # The worker records the stop without fitting.
else:
    run('fit', fold=3)
    print(json.dumps(receipt('fold_3/summary.json')['metrics'], indent=2))

## 3. Fresh-process replay and pooled review

Reload completed models and require exact saved predictions with **zero refits**. Pooled RMSE is computed from all squared coordinate errors and row counts, not the mean of fold RMSE. The primary decision uses the later folds only. All-fold pooling is descriptive.

**Replication gate:** at least 1% pooled gain on the two later folds, negative adjusted upper paired-game difference bound, and improvement on both later folds. The adjustment is exploratory and does not remove adaptive research or prior reuse of games.

In [ ]:
run('replay')
r = receipt('replay.json')
assert r['new_coordinate_models'] == 0
summary = receipt('summary.json')
print(json.dumps({'status':summary['status'],'replication_gate':summary['feature_replication_gate_passed'],'later_only':summary['pools']['later_folds_only']}, indent=2))

In [ ]:
show(visuals.temporal_scores(KIT,OUT),'temporal_scores')
show(visuals.temporal_intervals(OUT),'temporal_intervals')
show(visuals.later_horizon(OUT),'later_horizon')
show(visuals.influence_ranges(OUT),'game_influence')

## 4. Export the aggregate report, then stop

The report excludes raw tracking, model weights, per-row keys/predictions/targets and feature matrices. Do not proceed to new features or an ensemble automatically. A pass earns review for attribution/integration; a failure earns diagnosis, not unchanged scaling.

In [ ]:
run('report')
print(OUT/'nfl_feature_round5_report.zip')

Save the notebooks, download **nfl_feature_round5_report.zip**, and stop the existing SageMaker space when finished. Do not delete the space or checkpoints.